**Research questions**

1. What is predictive of berm condition / intactness? 
2. What is predictive of berm impact on surrounding vegetation, 
3. What is the correspondence between these condition and vegetation response, if any?

Methods: 
- Variables are highly correlated, we first examined which variables explain highest fraction variance in SAVI 



Results:
  - Longer berms more likely to be degraded. Perhaps because more failure points 


Landform is not the best predictor, but useful as a conceptual framework – correspondence to berm purpose (are some water spreader berms actually flood control berms in the dataset).
For managemen 

-  Fan terraces are more effective because steeper,  possibly because steeper (more runoff capture), possibly lower clay and no B horizon?

-  Floodplains less effective because gentle slopes,  possibly high clay

Different reasons, design criteria, inherent differnces.

Flow accumulation tends to be higher in floodplains --> intact berms tend to have lower flow accumulation

agregating over 6 years;
---

## Reference

Nichols, M. H., Duke, S. E., Holifield Collins, C., & Thompson, L. (2023). Legacy earthen berms influence vegetation and hydrologic complexity in the Altar Valley, Arizona. *International Soil and Water Conservation Research*. https://doi.org/10.1016/j.iswcr.2023.01.005


In [1]:
import pandas as pd
import glob
import numpy as np
import matplotlib.ticker as mtick
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import fisher_exact, chi2_contingency

import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module=r"seaborn(\..*)?")

# Global display preferences (apply notebook-wide)
from IPython.display import display, HTML
pd.set_option("display.max_columns", 50)


## Query soil data from API

In [2]:
import json
import re
import requests

URL = "https://sdmdataaccess.sc.egov.usda.gov/Tabular/post.rest"


def sda_post(sql: str, timeout=90) -> dict:
    payload = {"query": sql, "format": "JSON"}
    r = requests.post(
        URL,
        data=json.dumps(payload),
        headers={"Content-Type": "application/json"},
        timeout=timeout,
    )
    r.raise_for_status()

    txt = r.text or ""
    if len(txt.strip()) == 0:
        raise RuntimeError(
            "SDA returned an empty response body (timeout/limits). Try again or simplify query.")
    if txt.lstrip().startswith("<"):
        raise RuntimeError(f"SDA returned non-JSON (starts with '<'). First 400 chars:\n{txt[:400]}")
    return r.json()

def sda_to_df(resp: dict) -> pd.DataFrame:
    table_key = next((k for k, v in resp.items() if isinstance(v, list)), None)
    if table_key is None:
        raise RuntimeError(f"Unexpected SDA JSON keys: {list(resp.keys())}")
    return pd.DataFrame(resp[table_key])

def norm_landform(s) -> str:
    if s is None:
        return "unknown"
    s = str(s).strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s if s else "unknown"

# your one explicit recode
RECODE_EXACT = {"basin floors, fan terraces": "fan terraces"}

def canonical_landform(lf: str) -> str:
    lf = norm_landform(lf)
    return RECODE_EXACT.get(lf, lf)


def classify_landform(lf: str) -> str:
    lf = canonical_landform(lf)

    # Priority 1: stream terraces (always its own class)
    if ("stream terraces" in lf) :
        return "Stream terraces"

    # Priority 2: fans + fan terraces (incl alluvial fans + fan remnants)
    if ("fan terraces" in lf) or ("alluvial fans" in lf) or ("fan remnants" in lf) \
      or ("eroded fan" in lf) or ("fans" in lf) or ("fan piedmonts" in lf):
        return "Fan terraces"

    # Priority 3: valley bottom (basin floors + flood plains + drainageways/swales)
    if ("flood plains" in lf) or ("basin floors" in lf) or ("drainageways" in lf) or \
       ("swales" in lf) or ("plains" in lf):
        return "Flood plains"

    # Priority 4: uplands (hills/mountains/pediments + combos)
    if ("hills" in lf) or ("mountains" in lf) or ("pediments" in lf) or ("ridges" in lf):
        return "Fan terraces"# "Uplands"

    if ("terraces" in lf) :
        return "Stream terraces"    
    
    return  "Unknown"


def print_gee_dict(varname: str, d: dict):
    print(f"\nvar {varname} = ee.Dictionary({{")
    for k in sorted(d.keys(), key=lambda x: int(x) if str(x).isdigit() else str(x)):
        v = str(d[k]).replace("'", "\\'")
        print(f"  '{k}': '{v}',")
    print("});")

# -------------------------
# Curated overrides (optional; keep if you trust these more than SDA)
# -------------------------
MUKEY_to_landform_base = {
    "53807": "stream terraces",
    "53875": "flood plains",
    "53793": "fan terraces",
    "53738": "fan terraces",
    "53806": "flood plains",
    "53895": "fan terraces",
    "53886": "fan terraces, hills",
    "53799": "stream terraces",
    "53759": "flood plains",
}

mukeys_extra = [
    "1416070","1421630","1421631","1423101","1425287","1425309","1425386","1425394",
    "1425688", "1426107","1426108","1426109", "1427770","1427772","1427773","1427775", 
    "1427776","1427809","1427812","1427814","1427815","1427817","54393","54430",
    "53533","53539","53540","53546","53550","53557","53562","53563","53569",
    "53739","53741","53744","53750","53754","53756","53757","53767","53768",
    "53781","53783","53786","53788","53795","801772","801773","801777",
    "53808","53810","53812","53816","53817","53820","53822","53823","53824","53826",
    "53830","53832","53834","53838","53839","53840","53827","53863","54658","54594",
    "53844","53845","53846","53849","53851","53853","53855","53857","53859","53861",
    "53864","53865","53867","53868","53869","53876","53879","53881","53883","53885",
    "53891","53892","53897","53898","53899","53901","53902","53903","53907","53910",
    "54340","54341","54342","54343","54344","54345","54347","54365","53887","54364",
    "54354","54355","54356","54357","54358","54359","54360","54361","54362","54363",
    "54367","54370","54372","54374","54375","54376","54377","54378","54379","54380",
    "54382","54383","54384","54385","54386","54387","54389","54390","54391","54392",
    "54397","54398","54399","54400","54401","54402","54426","54427","54428","54429",
    "54404","54405","54406","54407","54408","54409","54410","54412","54413","54395",
    "54415","54416","54417","54418","54419","54420","54421","54423","54424","54654",
    "54538","54544","54546","54553","54555","54557","54561","54567","54582","54585",
    "54601","54609","54612","54616","54626","54628","54632","54635","54639","54645",
    "54661","54672",

]

mukeys = sorted(set(list(MUKEY_to_landform_base.keys()) + mukeys_extra), key=int)
in_list = ",".join(f"'{m}'" for m in mukeys)



def fetch_landforms_batch(batch_mukeys):
    in_list = ",".join(f"'{m}'" for m in batch_mukeys)
    sql = f"""
    WITH dom AS (
      SELECT
        c.mukey, c.cokey, c.compname, c.comppct_r,
        ROW_NUMBER() OVER (PARTITION BY c.mukey ORDER BY c.comppct_r DESC) AS rn
      FROM component c
      WHERE c.mukey IN ({in_list})
    )
    SELECT
      m.mukey,
      l.areasymbol,
      m.musym,
      m.muname,
      d.compname,
      d.comppct_r,
      cg.geomfname AS landform
    FROM mapunit m
    JOIN legend l ON l.lkey = m.lkey
    JOIN dom d ON d.mukey = m.mukey AND d.rn = 1
    LEFT JOIN cogeomordesc cg
      ON cg.cokey = d.cokey
     AND cg.geomftname = 'Landform'
     AND cg.rvindicator = 'yes'
    ORDER BY m.mukey;
    """
    resp = sda_post(sql)
    dfb = sda_to_df(resp).rename(columns={
        0: "mukey",
        1: "areasymbol",
        2: "musym",
        3: "muname",
        4: "compname",
        5: "comppct_r",
        6: "landform",

    })
    dfb["mukey"] = dfb["mukey"].astype(str)
    dfb["landform"] = dfb["landform"].apply(norm_landform)
    return dfb

BATCH_SIZE = 150
df_parts = []
for i in range(0, len(mukeys), BATCH_SIZE):
    df_parts.append(fetch_landforms_batch(mukeys[i:i + BATCH_SIZE]))

df = pd.concat(df_parts, ignore_index=True)

# collapse multiple landforms per MUKEY (if any)
mukey_landforms_sda = (
    df.dropna(subset=["landform"])
      .groupby("mukey")["landform"]
      .apply(lambda s: ", ".join(sorted(set(s.astype(str)))))
      .to_dict()
 )

# raw landforms (canonicalized) for all requested mukeys
MUKEY_to_landform_raw = {mk: canonical_landform(mukey_landforms_sda.get(mk, "unknown")) for mk in mukeys}
MUKEY_to_landform_raw.update({k: canonical_landform(v) for k, v in MUKEY_to_landform_base.items()})  # curated override

# THIS is the simplified MUKEY_to_landform you want
MUKEY_to_landform = {mk: classify_landform(lf) for mk, lf in MUKEY_to_landform_raw.items()}

print(pd.Series(MUKEY_to_landform).value_counts())


Fan terraces       145
Flood plains        35
Stream terraces     17
Unknown              4
Name: count, dtype: int64


In [3]:
def fetch_landforms_texture_batch(batch_mukeys):
    in_list = ",".join(f"'{m}'" for m in batch_mukeys)


    sql = f"""
    WITH dom AS (
      SELECT
        c.mukey, c.cokey, c.compname, c.comppct_r,
        ROW_NUMBER() OVER (PARTITION BY c.mukey ORDER BY c.comppct_r DESC) AS rn
      FROM component c
      WHERE c.mukey IN ({in_list})
    ),
    surf_hz AS (
      -- pick the topmost horizon for the dominant component
      SELECT
        ch.cokey, ch.chkey, ch.hzdept_r, ch.hzdepb_r,
        ch.sandtotal_r, ch.silttotal_r, ch.claytotal_r,
        ROW_NUMBER() OVER (
          PARTITION BY ch.cokey
          ORDER BY
            CASE WHEN ch.hzdept_r = 0 THEN 0 ELSE 1 END,
            ch.hzdept_r
        ) AS rn
      FROM chorizon ch
      JOIN dom d ON d.cokey = ch.cokey AND d.rn = 1
      WHERE ch.hzdept_r IS NOT NULL
        AND ch.hzdepb_r IS NOT NULL
        AND ch.hzdepb_r > 0
    ),
    tex AS (
      -- representative texture class for that surface horizon
      SELECT
        t0.chkey,
        t0.texcl
      FROM (
        SELECT
          chtg.chkey,
          ct.texcl,
          ROW_NUMBER() OVER (
            PARTITION BY chtg.chkey
            ORDER BY
              CASE WHEN chtg.rvindicator = 'yes' THEN 0 ELSE 1 END,
              ct.texcl
          ) AS rn
        FROM chtexturegrp chtg
        JOIN chtexture ct ON ct.chtgkey = chtg.chtgkey
      ) t0
      WHERE t0.rn = 1
    )
    SELECT
      m.mukey,
      l.areasymbol,
      m.musym,
      m.muname,
      d.compname,
      d.comppct_r,
      cg.geomfname AS landform,
      hz.sandtotal_r,
      hz.silttotal_r,
      hz.claytotal_r,
      tx.texcl
    FROM mapunit m
    JOIN legend l ON l.lkey = m.lkey
    JOIN dom d ON d.mukey = m.mukey AND d.rn = 1
    LEFT JOIN cogeomordesc cg
      ON cg.cokey = d.cokey
     AND cg.geomftname = 'Landform'
     AND cg.rvindicator = 'yes'
    LEFT JOIN surf_hz hz
      ON hz.cokey = d.cokey AND hz.rn = 1
    LEFT JOIN tex tx
      ON tx.chkey = hz.chkey
    ORDER BY m.mukey;
    """

    resp = sda_post(sql)
    dfb = sda_to_df(resp).rename(columns={
        0: "mukey",
        1: "areasymbol",
        2: "musym",
        3: "muname",
        4: "compname",
        5: "comppct_r",
        6: "landform",
        7: "sandtotal_r",
        8: "silttotal_r",
        9: "claytotal_r",
        10: "texcl",
    })

    dfb["mukey"] = dfb["mukey"].astype(str)
    dfb["landform"] = dfb["landform"].apply(norm_landform)
    return dfb

BATCH_SIZE = 150
df_parts = []
for i in range(0, len(mukeys), BATCH_SIZE):
    df_parts.append(fetch_landforms_texture_batch(mukeys[i:i + BATCH_SIZE]))
df = pd.concat(df_parts, ignore_index=True)

df['sandtotal_r'] = df['sandtotal_r'].astype(float)
df['claytotal_r'] = df['claytotal_r'].astype(float)
df['silttotal_r'] = df['silttotal_r'].astype(float)

MUKEY_to_texcl = df.set_index("mukey")["texcl"].fillna("unknown").to_dict()

MUKEY_to_clay = df.set_index("mukey")["claytotal_r"].to_dict()  # numeric %, may be NaN
MUKEY_to_sand = df.set_index("mukey")["sandtotal_r"].to_dict()
MUKEY_to_silt = df.set_index("mukey")["silttotal_r"].to_dict()

In [4]:
df["muname_base"] = (
    df["muname"]
      .astype(str)
      .str.split(",", n=1).str[0]
      .str.strip()
)
df["muname_base"].value_counts().head(30)


slope_re = re.compile(r"(\d+)\s*(?:to|-)\s*(\d+)\s*percent", re.I)

def parse_slope_range(s):
    m = slope_re.search(str(s))
    if not m:
        return (np.nan, np.nan)
    return (float(m.group(1)), float(m.group(2)))

df[["slope_lo", "slope_hi"]] = df["muname"].apply(parse_slope_range).apply(pd.Series)

In [5]:
# 1) clean the name (drop slope text after comma; drop trailing MLRA notes)
df = df.copy()
df["muname_core"] = (
    df["muname"].astype(str).str.lower()
      .str.split(",").str[0]
      .str.replace(r"\s+mlra.*$", "", regex=True)
      .str.replace(r"\s+", " ", regex=True)
      .str.strip()
)

# 2) texture terms (order matters: longest first)
texture_terms = [
    "very fine sandy loam",
    "fine sandy loam",
    "loamy coarse sand",
    "loamy sand",
    "sandy clay loam",
    "silty clay loam",
    "clay loam",
    "silt loam",
    "sandy loam",
    "sandy clay",
    "silty clay",
    "clay",
    "silt",
    "loam",
    "sand",
]
tex_re = r"\b(" + "|".join(map(re.escape, texture_terms)) + r")\b"

# 3) optional coarse-fragment modifiers right before the texture term
mods_re = r"(?P<mods>(?:\b(?:extremely|very)?\s*(?:gravelly|cobbly|stony|channery|shaly)\b\s*)+)?"

pat = mods_re + r"(?P<texture>" + tex_re + r")"

ex = df["muname_core"].str.extract(pat)

df["texture_class"] = ex["texture"].str.strip().str.replace(r"\s+", " ", regex=True).str.title()


## Load CSVs in data folder (GEE exports)

In [6]:
# --- 1) Load and combine CSV files --------------------------------------------------
# Find all CSVs matching the pattern (e.g., AOI01_bermdata_20250205.csv) in the folder.

file_paths = sorted(glob.glob('../data/berm_exports/AOI*_bermdata_*0130.csv'))

# Read each CSV into a DataFrame.
df_list = [pd.read_csv(fp) for fp in file_paths]

# Concatenate all AOI data into a single DataFrame (resetting the row index).
data = pd.concat(df_list, ignore_index=True)


# --- 2) MUKEY mapping dictionaries --------------------------------------------------
# These dicts translate MUKEY (soil map unit keys) to human-readable attributes.
# Note: MUKEYs are strings; keep keys as strings to avoid mismatches later.

MUKEY_to_mapunitname = {
    '53807': 'Glendale silt loam, 0 to 3 percent slopes',
    '53875': 'Riveroad and Comoro soils, 0 to 2 percent slopes',
    '53793': 'Diaspar sandy loam, 1 to 5 percent slopes',
    '53738': 'Altar-Sasabe complex, 1 to 8 percent slopes',
    '53806': 'Anthony fine sandy loam, 0 to 3 percent slopes',
    '53895': 'Bucklebar-Sahuarita complex, 0 to 3 percent slopes',
    '53886': 'Sasabe-Caralampi complex, 1 to 15 percent slopes',
    '53799': 'Glendale-Bucklebar complex, 0 to 3 percent slopes',
    '53759': 'Comoro sandy loam, 0 to 2 percent slopes'
}

MUKEY_to_slopeclass = {
    '53807': 'low', '53875': 'low', '53793': 'high', '53738': 'high',
    '53806': 'low', '53895': 'low', '53886': 'high', '53799': 'low', 
    '53759': 'low'
}

MUKEY_to_parentmaterial = {
    '53807': 'Mixed alluvium',
    '53875': 'Moderately fine textured alluvium, Moderately coarse textured alluvium',
    '53793': 'Alluvium derived from granite and/or alluvium derived from schist',
    '53738': 'Alluvium derived from schist and/or alluvium derived from granite, Mixed alluvium',
    '53806': 'Mixed alluvium', 
    '53895': 'Mixed alluvium', 
    '53886': 'Mixed alluvium',
    '53799': 'Mixed alluvium', 
    '53759': 'Mixed alluvium'
}


MUKEY_to_typicalprofile = {
    '53807': 'A-C', '53875': 'A-C, A-C1-C2-C3', '53793': 'A-Bt-2Bt-3BCt',
    '53738': 'A-Bw-BC-C, A-Bt1-Bt2-Bt3-2Btk', '53806':  'A-AC-C-Ck-C',
    '53895': 'A/Bt-Bt-Btk, A-Bk-2Btkb1-2Btkb2', '53886': 'A-Bt1-Bt2-Bt3-Btk, A-Bt1-Bt2-2C',
    '53799': 'A-C, A/Bt-Bt-Btk', '53759': 'A1-A2-C1-C2'
}



In [7]:
# Rebuild MUKEY -> typical horizon sequence(s) from Soil Data Access
mukeys = sorted(data["MUKEY"].dropna().astype(str).unique())
BATCH_SIZE = 150

def fetch_horizons_for_mukeys(batch_mukeys):
    in_list = ",".join(f"'{m}'" for m in batch_mukeys)
    sql = f"""
    SELECT
      c.mukey,
      c.cokey,
      c.compname,
      c.comppct_r,
      c.majcompflag,
      h.hzname,
      h.hzdept_r
    FROM component c
    JOIN chorizon h ON h.cokey = c.cokey
    WHERE c.mukey IN ({in_list})
      AND h.hzname IS NOT NULL
    ORDER BY c.mukey, c.comppct_r DESC, c.cokey, h.hzdept_r;
    """
    resp = sda_post(sql)
    dfb = sda_to_df(resp).rename(columns={
        0: "mukey",
        1: "cokey",
        2: "compname",
        3: "comppct_r",
        4: "majcompflag",
        5: "hzname",
        6: "hzdept_r",
    })
    dfb["mukey"] = dfb["mukey"].astype(str)
    return dfb

df_parts = []
for i in range(0, len(mukeys), BATCH_SIZE):
    df_parts.append(fetch_horizons_for_mukeys(mukeys[i:i+BATCH_SIZE]))

hz = pd.concat(df_parts, ignore_index=True)

# Choose which components to include:
# Option A: only major components (often matches "multiple profiles per MUKEY")
hz_use = hz[hz["majcompflag"].str.lower().eq("yes")].copy()

# If that yields empty for some MUKEYs, fall back to dominant component:
if hz_use.empty:
    hz_use = hz.copy()

hz_use = hz_use.sort_values(["mukey", "comppct_r", "cokey", "hzdept_r"],
                            ascending=[True, False, True, True])

# Horizon sequence per component
comp_profiles = (
    hz_use.groupby(["mukey", "cokey"], as_index=False)
          .agg(profile=("hzname", lambda s: "-".join(s.astype(str))))
)

# Join unique component profiles per MUKEY
MUKEY_to_typicalprofile_auto = (
    comp_profiles.groupby("mukey")["profile"]
                 .apply(lambda s: ", ".join(pd.unique(s)))
                 .to_dict()
)

MUKEY_to_typicalprofile = MUKEY_to_typicalprofile_auto

# --- 3) Standardize MUKEY and apply mappings ---------------------------------------
# Ensure MUKEY is string so dictionary lookups work even if the source was numeric.
data['MUKEY'] = data['MUKEY'].astype(str)
# Robust MUKEY normalization: keeps missing as <NA> and removes .0 issues

data['MapUnitName'] = data['MUKEY'].map(MUKEY_to_mapunitname)
data['SlopeClass'] = data['MUKEY'].map(MUKEY_to_slopeclass)
data['Landform'] = data['MUKEY'].map(MUKEY_to_landform)
data['ParentMaterial'] = data['MUKEY'].map(MUKEY_to_parentmaterial)
data['TypicalProfile'] = data['MUKEY'].map(MUKEY_to_typicalprofile)
data['Texture'] = data['MUKEY'].map(MUKEY_to_texcl)

data['claytotal_r'] = data['MUKEY'].map(MUKEY_to_clay)
data['sandtotal_r'] = data['MUKEY'].map(MUKEY_to_sand)
data['silttotal_r'] = data['MUKEY'].map(MUKEY_to_silt)

# Harmonize landform label 
data['Landform'] = data['Landform'].replace({"Fan terraces, hills": "Fan terraces"})

# Check for unmatched MUKEYs 
# Inspect MUKEYs that did not find a match in the dictionaries—useful QA check.
unmatched_keys = data[data['MapUnitName'].isna()]['MUKEY'].unique()
print(f"Unmatched MUKEYs: {unmatched_keys}")

# Landform is from MUKEYS

Unmatched MUKEYs: ['53844' '53897' '53817' '53855' '53899' '53744' '53820' '53849' '53757'
 '53902' '53907' '53865' '53822' '53826' '53846' '53783' '53901' '53788'
 '53812' '53879' '53827' '1423101' '53756' '53864']


In [8]:
# --- Classify berm condition -----------------------------------------------------
# Fill missing failure types as "Intact" and collapse to a binary condition:
# "Intact" vs "Degraded" (anything not exactly "Intact" is considered degraded).
data['Fail_Type'] = data['Fail_Type'].fillna("Intact")
data['Condition'] = data['Fail_Type'].apply(lambda x: 'Intact' if x == 'Intact' else 'Degraded')
data = data.loc[data.query("Fail_Type != 'Breach and Flank'").index]

# --- 5) Group and summarize ---------------------------------------------------------
# Count berms by Landform x Condition and display as a contingency table.
summary_counts = data.groupby(['Landform', 'Condition']).size().unstack(fill_value=0)
print("\nBerm Integrity Counts by Landform:\n")
print(summary_counts)


# Count berms by Landform x Condition and display as a contingency table.
summary_counts = data.groupby(['Landform', 'Fail_Type']).size().unstack(fill_value=0)
print("\nBerm Integrity Counts by Landform:\n")
print(summary_counts)

# Convert counts to within-landform proportions (rows sum to 1), rounded to 2 decimals.
summary_proportions = summary_counts.div(summary_counts.sum(axis=1), axis=0).round(2)
print("\nProportion of Berm Conditions by Landform:\n")
print(summary_proportions)

data['savi_background'] = data['savi_background'].replace(0, np.nan)

data['effect'] = (data['saviU_60'] - data['saviD_60'])  / data['savi_background']
data['effect_percent'] = data['effect'] * 100


data['Landform'] = data['Landform'].replace({ "Fan terraces, hills": "Fan terraces"})


Berm Integrity Counts by Landform:

Condition        Degraded  Intact
Landform                         
Fan terraces          123     153
Flood plains          144     215
Stream terraces        50      90

Berm Integrity Counts by Landform:

Fail_Type        Breach  Flank  Intact
Landform                              
Fan terraces         31     92     153
Flood plains         64     80     215
Stream terraces      19     31      90

Proportion of Berm Conditions by Landform:

Fail_Type        Breach  Flank  Intact
Landform                              
Fan terraces       0.11   0.33    0.55
Flood plains       0.18   0.22    0.60
Stream terraces    0.14   0.22    0.64


In [9]:
# Define berm length category
data = data.dropna(subset=['Shape_Leng', 'Fail_Type'])

data['Berm_Length_Class'] = data['Shape_Leng'].apply(
    lambda x: 'Short (≤ 50 m)' if x <= 50 else 'Long (> 50 m)'
)

# Bin slope
data['Slope_Class'] = data['slope_100'].apply(
    lambda x: "Shallow (≤ 2%)" if x <= 2 else "Steep (> 2%)"
)

data['Soil_Development'] = data['TypicalProfile'].astype(str).apply(
    lambda x: 'B horizon' if 'B' in x else 'No B horizon'
)

data['effective'] = False
data.loc[data['effect_percent'] > 7, 'effective'] =  True
data.loc[data['effect_percent'] < 7, 'effective'] = False

data['Effective'] = ''
data.loc[data['effect_percent'] > 7, 'Effective'] =  'Effective'
data.loc[data['effect_percent'] < 7, 'Effective']=  'Ineffective'

data['High_Clay'] = data['claytotal_r'] > data['claytotal_r'].median()
data['Intact'] = data['Condition'] == 'Intact'



In [10]:
# landform : from my dataset
data['proximity'] = data['landform'].replace({
    0 : 'Upland',
    1 : 'Flood plain'    
})


In [11]:
# data.groupby("Texture")['effective'].mean(),\
data.query("Fail_Type =='Breach'").groupby("Texture")['effective'].mean()
# Where breaches occurred on coarser textured soils, the  up and downslope differences in grass cover diminished.



Texture
Clay loam          0.471698
Fine sandy loam    0.625000
Loam               1.000000
Loamy sand         0.500000
Sandy clay loam    0.000000
Sandy loam         0.428571
Silt loam          0.578947
Name: effective, dtype: float64

In [12]:
# Keep textures with >=10 samples in each Intact category, then summarize
sub = data.dropna(subset=["Texture", "Intact", "effective"]).copy()

counts = sub.groupby(["Texture", "Intact"]).size().unstack(fill_value=0)
valid_textures = counts[(counts.min(axis=1) >= 10)].index

texture_order = ["Clay loam",  "Silt loam",  "Loam", "Fine sandy loam", "Sandy loam"]

out = (
    sub[sub["Texture"].isin(valid_textures)]
      .groupby(["Texture", "Intact"])["effective"]
      .agg(mean="mean", count="size")
      .reset_index()
      .assign(Texture=lambda d: pd.Categorical(d["Texture"], categories=texture_order, ordered=True))
      .sort_values(["Texture", "Intact"])
      .reset_index(drop=True)
)

# intact berms - more effective than degraded berms on coarser soils (sandy loam, fine sandy loam)

In [13]:
# For each texture: test whether effectiveness differs between Intact vs Degraded berms
sub = data.dropna(subset=["Texture", "Intact", "effective"]).copy()

# Optional guard to avoid testing very sparse categories
MIN_N_PER_CATEGORY = 10

# Mean/count summary by texture and intact status
summary = (
    sub.groupby(["Texture", "Intact"])["effective"]
       .agg(mean="mean", count="size")
       .reset_index()
       .assign(Group=lambda d: d["Intact"].map({False: "Degraded", True: "Intact"}))
       .drop(columns="Intact")
       .sort_values(["Texture", "Group"])
       .reset_index(drop=True)
)

rows = []
for texture, g in sub.groupby("Texture"):
    ct = pd.crosstab(g["Intact"], g["effective"]).reindex(
        index=[False, True], columns=[False, True], fill_value=0
    )

    n_degraded = int(ct.loc[False].sum())
    n_intact = int(ct.loc[True].sum())
    p_degraded = (ct.loc[False, True] / n_degraded) if n_degraded > 0 else np.nan
    p_intact = (ct.loc[True, True] / n_intact) if n_intact > 0 else np.nan

    if min(n_degraded, n_intact) >= MIN_N_PER_CATEGORY:
        table = [
            [ct.loc[False, False], ct.loc[False, True]],
            [ct.loc[True,  False], ct.loc[True,  True]],
        ]
        odds_ratio, p_value = fisher_exact(table, alternative="two-sided")
        tested = True
    else:
        odds_ratio, p_value = np.nan, np.nan
        tested = False

    rows.append({
        "Texture": texture,
        "n_degraded": n_degraded,
        "n_intact": n_intact,
       "p_effective_degraded": p_degraded,
       "p_effective_intact": p_intact,
       # "odds_ratio": odds_ratio,
        "p_value": p_value,
        "significant_0p05": (p_value < 0.05) if pd.notna(p_value) else False
    })

results = (
    pd.DataFrame(rows)
      .sort_values([ "p_value", "Texture"], ascending=[ True, True])
      .reset_index(drop=True)
)

display(summary)
display(results)
# on corase soils, intact berms tend to be more effective.
# on finer soils, intact berms tend to be less effective, but the difference is not significant.

,Texture,mean,count,Group
0,Clay loam,0.416667,108,Degraded
1,Clay loam,0.377143,175,Intact
2,Fine sandy loam,0.571429,21,Degraded
3,Fine sandy loam,0.363636,33,Intact
4,Loam,0.571429,14,Degraded
5,Loam,0.289474,38,Intact
6,Loamy coarse sand,0.400000,5,Degraded
7,Loamy coarse sand,0.250000,4,Intact
8,Loamy sand,0.200000,5,Degraded
9,Loamy sand,0.333333,3,Intact


,Texture,n_degraded,n_intact,p_effective_degraded,p_effective_intact,p_value,significant_0p05
0,Sandy loam,113,97,0.530973,0.680412,0.034068,True
1,Loam,14,38,0.571429,0.289474,0.103020,False
2,Fine sandy loam,21,33,0.571429,0.363636,0.166631,False
3,Clay loam,108,175,0.416667,0.377143,0.532521,False
4,Silt loam,49,88,0.510204,0.556818,0.720822,False
5,Loamy coarse sand,5,4,0.400000,0.250000,NaN,False
6,Loamy sand,5,3,0.200000,0.333333,NaN,False
7,Sandy clay loam,2,20,0.500000,0.400000,NaN,False


In [14]:
import sys as _sys
_sys.path.insert(0, '../src')
from constants import (
    INTACT_COL, DEGRADED_COL, BREACH_COL, FLANK_COL,
    LF_COLORS, lf_order,
    LENGTH_COLORS, length_order,
    SLOPE_COLORS, slope_order,
    CLAY_COLORS, clay_order,
    SOILDEV_COLORS, soildev_order,
    TEXTURE_COLORS, texture_order,
    LBL_EFFECTIVE, LBL_INEFFECTIVE, eff_order, eff_colors,
    fail_order, fail_colors,
    MODEL_CLR_CONDITION, MODEL_CLR_VEGRESPONSE, MODEL_CLR_CHANCE,
)

# Align Effective column values with canonical labels from constants.py
data['Effective'] = data['Effective'].map({
    'Effective':   LBL_EFFECTIVE,
    'Ineffective': LBL_INEFFECTIVE,
})

In [15]:

sub = data.query("Type == 'Berm'").copy() if "Type" in data.columns else data.copy()
sub = sub.dropna(subset=["Fail_Type", "Texture"])

# 1) Counts: Texture × Fail_Type
ct = pd.crosstab(sub["Texture"], sub["Fail_Type"])
ct_t = ct.copy()
ct_t['Total'] = np.sum(ct_t, axis=1)

# 2) Within-texture proportions (rows sum to 1)
ct_row = ct.div(ct.sum(axis=1), axis=0)
display((ct_row * 100).round(1))



Fail_Type,Breach,Flank,Intact
Texture,,,
Clay loam,20.7,19.9,59.3
Fine sandy loam,15.4,33.3,51.3
Loam,8.6,31.4,60.0
Loamy coarse sand,0.0,55.6,44.4
Loamy sand,28.6,28.6,42.9
Sandy clay loam,0.0,0.0,100.0
Sandy loam,13.2,41.5,45.4
Silt loam,12.0,27.0,61.0


In [16]:
# merge shapefile attrs into existing pandas df `data` on rounded lat/lon
# keep lower-case; do not bring over shapefile columns that duplicate names in `data`

import geopandas as gpd
import os
# --- config ---
shp_path  = "../data/Berm_Directionality/Berm_Directionality.shp"
out_csv   = "../data/merged.csv"
round_ndp = 6

# --- helpers ---
def _normalize_cols(df):
    df = df.copy()
    df.columns = [c.strip().replace(" ", "_") for c in df.columns]
    return df

def _find_lat_lon(cols):
    lc = {c.lower(): c for c in cols}
    lat_candidates = ["lat","latitude","y","lat_dd"]
    lon_candidates = ["lon","long","longitude","x","lon_dd"]
    lat = next((lc[c] for c in lat_candidates if c in lc), None)
    lon = next((lc[c] for c in lon_candidates if c in lc), None)
    return lat, lon

def _ll_key(df, lat_col, lon_col, ndp=6):
    lat = pd.to_numeric(df[lat_col], errors="coerce").round(ndp)
    lon = pd.to_numeric(df[lon_col], errors="coerce").round(ndp)
    return lat.astype("string") + "_" + lon.astype("string")

# --- require an existing `data` dataframe ---
if "data" not in globals():
    raise NameError("expected a pandas dataframe named `data` in memory")

data = _normalize_cols(data)
data_lat, data_lon = _find_lat_lon(data.columns)
if not data_lat or not data_lon:
    raise KeyError(f"couldn't find lat/lon in `data`; columns: {list(data.columns)}")

# read shapefile, normalize cols
gdf = gpd.read_file(shp_path)
if gdf.crs is None:
    gdf = gdf.set_crs(epsg=4326)
elif gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(epsg=4326)
gdf = _normalize_cols(gdf)

shp_lat, shp_lon = _find_lat_lon(gdf.columns)
if not shp_lat or not shp_lon:
    raise KeyError(f"couldn't find lat/lon in shapefile; columns: {list(gdf.columns)}")

# build merge keys
left  = data.copy()
right = gdf.drop(columns=["geometry"], errors="ignore").copy()

left["__ll_key__"]  = _ll_key(left,  data_lat, data_lon, ndp=round_ndp)
right["__ll_key__"] = _ll_key(right, shp_lat,  shp_lon,  ndp=round_ndp)

# keep only NON-OVERLAPPING shapefile columns (plus the join key)
cols_to_add = ["__ll_key__"] + [c for c in right.columns if c not in left.columns and c != "__ll_key__"]
right = right[cols_to_add]

# merge (no suffixes needed because we've removed duplicates by name)
merged = left.merge(right, on="__ll_key__", how="left").drop(columns="__ll_key__")

# save
os.makedirs(os.path.dirname(out_csv), exist_ok=True)
merged.to_csv(out_csv, index=False)
print(f"done. merged rows = {len(merged):,} -> {out_csv}")


done. merged rows = 775 -> ../data/merged.csv
